In [1]:
using Pkg
using Random
using LinearAlgebra
using Printf
using myExample
using Optimisers
using JLD2, Statistics

const AD = myExample.AutoDiff
const MF = myExample.MiniFlux

Random.seed!(123)

println("Loading dataset...")
data_path_prefix = "../data/"
try
    global X_train = load(joinpath(data_path_prefix, "imdb_dataset_prepared.jld2"), "X_train")
    global y_train = load(joinpath(data_path_prefix, "imdb_dataset_prepared.jld2"), "y_train")
    global X_test = load(joinpath(data_path_prefix, "imdb_dataset_prepared.jld2"), "X_test")
    global y_test = load(joinpath(data_path_prefix, "imdb_dataset_prepared.jld2"), "y_test")
    global embeddings = load(joinpath(data_path_prefix, "imdb_dataset_prepared.jld2"), "embeddings")
    global vocab = load(joinpath(data_path_prefix, "imdb_dataset_prepared.jld2"), "vocab")
catch e
    println("Error loading data: $e"); exit()
end

println("Dataset loaded. X_train size: $(size(X_train)), y_train size: $(size(y_train))")

embedding_dim = size(embeddings, 1)
vocab_size    = length(vocab)
max_seq_len   = size(X_train, 1)

W_out_conv = max_seq_len - 3 + 1
if W_out_conv < 8
    error("Output width after Conv1D ($W_out_conv) is less than MaxPool1D pool size (8).")
end
W_out_pool = div(W_out_conv - 8, 8) + 1
actual_dense_input_features = W_out_pool * 8

model = MF.chain(
    MF.Embedding(vocab_size, embedding_dim),
    x -> AD.PermuteDimsOp(x, (2, 1, 3)),
    MF.Conv1D((3,), embedding_dim => 8, activation=AD.relu),
    MF.MaxPool1D((8,)),
    MF.Flatten(),
    MF.Dense(actual_dense_input_features, 1, AD.σ)
)

if model.layers[1] isa MF.Embedding
    target_size = size(model.layers[1].W.output)
    source_size = size(embeddings)
    if target_size == source_size
        model.layers[1].W.output .= embeddings
        println("Embeddings assigned to model.")
    else
        println("Error: Embeddings matrix size $source_size does not match model embedding layer size $target_size.")
    end
else
    println("Error: First layer is not an Embedding layer.")
end

batchsize = 64
train_loader = MF.create_batches(X_train, y_train, batchsize)
test_loader = MF.create_batches(X_test, y_test, batchsize)

learning_rate = 0.003f0
opt_rule = Optimisers.Adam(learning_rate)
epochs = 5

MF.train_with_optimisers!(model, MF.binary_cross_entropy_loss,
                          train_loader, test_loader,
                          opt_rule, epochs)
println("Training finished."); flush(stdout)

Loading dataset...
Dataset loaded. X_train size: (130, 40000), y_train size: (1, 40000)
Embeddings assigned to model.

Starting training...

===== Starting Epoch 1/5 =====
Epoch 1, Batch 62/625: Loss: 1.0045, Time/Batch: 0.030s
Epoch 1, Batch 124/625: Loss: 0.8950, Time/Batch: 0.028s
Epoch 1, Batch 186/625: Loss: 0.7768, Time/Batch: 0.031s
Epoch 1, Batch 248/625: Loss: 0.7447, Time/Batch: 0.024s
Epoch 1, Batch 310/625: Loss: 0.6647, Time/Batch: 0.023s
Epoch 1, Batch 372/625: Loss: 0.7067, Time/Batch: 0.028s
Epoch 1, Batch 434/625: Loss: 0.6918, Time/Batch: 0.028s
Epoch 1, Batch 496/625: Loss: 0.7164, Time/Batch: 0.026s
Epoch 1, Batch 558/625: Loss: 0.6911, Time/Batch: 0.028s
Epoch 1, Batch 620/625: Loss: 0.6846, Time/Batch: 0.027s
Epoch 1, Batch 625/625: Loss: 0.6557, Time/Batch: 0.028s
Epoch 1: Calculating accuracies...
EPOCH 1 SUMMARY: Avg Loss: 0.8230, Train Acc: 63.79%, Val Acc: 58.80%, Epoch Time: 35.23s (Avg Batch: 0.056s)
===== Starting Epoch 2/5 =====
Epoch 2, Batch 62/625: Los